# 1. Datasets FW.Br and FT.Br (Section 3.1, Table 1, Figure 2)

Builds the two datasets used in the paper from the raw exports in `datasets/`:

1. remove the excluded member IDs, messages without text and "trava-zap" messages;
2. preprocess the text;
3. keep messages with at least one religious keyword;
4. drop messages with any election-campaign keyword.

The filtered datasets are saved in `datasets/processed/` and reused by the other notebooks.
All parameters are in `utils.py`.

In [ ]:
from utils import *
import matplotlib.pyplot as plt
%matplotlib inline

## Filtering

In [ ]:
datasets, steps = {}, {}
for platform in ["whatsapp", "telegram"]:
    datasets[platform], steps[NAMES[platform]] = build_dataset(platform)
    save_dataset(datasets[platform], platform)

pd.DataFrame(steps)

## Table 1: basic statistics

Unique messages are distinct texts; tokens are counted on the preprocessed text of the unique messages.

In [ ]:
def basic_statistics(df):
    unique = df.drop_duplicates(subset=["text_content_anonymous"])
    tokens = unique["text_processed"].str.split().str.len()
    return {"messages": len(df), "unique messages": len(unique),
            "tokens mean": round(tokens.mean(), 2), "tokens std": round(tokens.std(), 2),
            "tokens min": tokens.min(), "tokens max": tokens.max()}

table1 = pd.DataFrame({NAMES[p]: basic_statistics(df) for p, df in datasets.items()})
table1.to_csv(results_folder("01_datasets") / "table1_statistics.csv")
table1

## Figure 2: messages per day

In [ ]:
per_day = pd.DataFrame({
    p: pd.to_datetime(df["date_message"]).dt.floor("D").value_counts() for p, df in datasets.items()
}).sort_index().asfreq("D").fillna(0)

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(per_day.index, per_day["whatsapp"], marker="o", label="WhatsApp")
ax.plot(per_day.index, per_day["telegram"], marker="x", linestyle="--", label="Telegram")
ax.set(title="Number of Messages per Day (WhatsApp vs. Telegram)", xlabel="Date", ylabel="Number of Messages")
ax.legend(); ax.grid(True); plt.xticks(rotation=45); plt.tight_layout()
fig.savefig(results_folder("01_datasets") / "figure2_messages_per_day.png", dpi=200)
plt.show()

## Comparison with the paper

In [ ]:
w, t = steps["FW.Br"], steps["FT.Br"]
compare_with_paper([
    ("FW.Br raw messages", 598971, w["raw messages"]),
    ("FW.Br raw users", 15286, w["raw users"]),
    ("FW.Br raw groups", 240, w["raw groups"]),
    ("FW.Br messages after filters", 5167, w["after exclusion filter (political)"]),
    ("FW.Br unique messages", 4329, table1.loc["unique messages", "FW.Br"]),
    ("FW.Br tokens mean", 82.34, table1.loc["tokens mean", "FW.Br"]),
    ("FW.Br tokens std", 99.64, table1.loc["tokens std", "FW.Br"]),
    ("FW.Br tokens max", 1683, table1.loc["tokens max", "FW.Br"]),
    ("FT.Br raw messages", 557586, t["raw messages"]),
    ("FT.Br raw users", 14810, t["raw users"]),
    ("FT.Br raw groups", 178, t["raw groups"]),
    ("FT.Br messages after filters", 8526, t["after exclusion filter (political)"]),
    ("FT.Br unique messages", 6462, table1.loc["unique messages", "FT.Br"]),
    ("FT.Br tokens mean", 88.57, table1.loc["tokens mean", "FT.Br"]),
    ("FT.Br tokens std", 116.87, table1.loc["tokens std", "FT.Br"]),
    ("FT.Br tokens max", 1549, table1.loc["tokens max", "FT.Br"]),
])